In [0]:
from pyspark.sql.functions import col

df_bronze = spark.table(
    "practice_rg.default.bronze_sales"
)

df_typed = (
    df_bronze
    .withColumn("order_id", col("order_id").cast("int"))
    .withColumn("quantity", col("quantity").cast("int"))
    .withColumn("price", col("price").cast("double"))
)

df_valid = df_typed.filter(
    col("order_id").isNotNull()
    & col("product").isNotNull()
    & (col("product") != "")
    & col("city").isNotNull()
    & (col("city") != "")
    & col("quantity").isNotNull()
    & (col("quantity") > 0)
    & col("price").isNotNull()
    & (col("price") >= 0)
)

df_silver = df_valid.dropDuplicates(["order_id"])

(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("practice_rg.default.silver_sales")
)

display(spark.table("practice_rg.default.silver_sales"))